In [18]:
# import a utility function for loading Roboflow models
from inference import get_model

# define the local image path to use for inference
image = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/roboflow_dataset_2/train/images/country1_000053_jpg.rf.3cbf340cc0f618ff3a4932e3f39e301a.jpg"

# load the road damage detection model
model = get_model(model_id="road-damage-5lxtz-zqhl5/3", api_key="IJU8d9jXU38Ts1Fqoev2")

# run inference on our chosen image, image can be a url, a numpy array, a PIL image, etc.
results = model.infer(image)

In [19]:
results[0].predictions

[ObjectDetectionPrediction(x=436.02745819091797, y=533.3561096191406, width=397.03199768066406, height=211.97540283203125, confidence=0.8839954733848572, class_name='1', class_confidence=None, class_id=1, tracker_id=None, detection_id='b0be3eab-8ac0-499c-ad3d-fb765b643f2f', parent_id=None),
 ObjectDetectionPrediction(x=353.0489044189453, y=526.8802795410156, width=76.68136596679688, height=221.35589599609375, confidence=0.7448828816413879, class_name='3', class_confidence=None, class_id=3, tracker_id=None, detection_id='b42000a5-dbdd-4df6-829c-6e38037224a0', parent_id=None)]

In [20]:
results[0].image.width, results[0].image.height

(640, 640)

In [21]:
def convert_results_to_yolo_format(results):
    """
    Convert Roboflow inference results to YOLO format string.
    
    Args:
        results: Roboflow inference results object
        
    Returns:
        str: YOLO format string with format:
             <class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>
    """
    if not results or len(results) == 0:
        return ""
    
    # Get image dimensions
    img_width = results[0].image.width
    img_height = results[0].image.height
    
    yolo_lines = []
    
    # Process each prediction
    for prediction in results[0].predictions:
        # Extract values
        class_id = prediction.class_id
        x_center = prediction.x
        y_center = prediction.y
        width = prediction.width
        height = prediction.height
        confidence = prediction.confidence
        
        # Normalize coordinates (convert to 0-1 range)
        x_center_normalized = x_center / img_width
        y_center_normalized = y_center / img_height
        width_normalized = width / img_width
        height_normalized = height / img_height
        
        # Format as YOLO string
        yolo_line = f"{class_id} {x_center_normalized:.6f} {y_center_normalized:.6f} {width_normalized:.6f} {height_normalized:.6f} {confidence:.6f}"
        yolo_lines.append(yolo_line)
    
    return "\n".join(yolo_lines)

# Test the function with the current results
yolo_format_string = convert_results_to_yolo_format(results)
print("YOLO Format Output:")
print(yolo_format_string)

YOLO Format Output:
1 0.681293 0.833369 0.620362 0.331212 0.883995
3 0.551639 0.823250 0.119815 0.345869 0.744883


In [22]:
import os
import glob
from pathlib import Path
from tqdm import tqdm

# Configuration
TEST_DATA_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/test_data"
OUTPUT_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

def get_all_test_images():
    """Collect all test images from all countries"""
    all_images = []
    
    for country in ['country_1', 'country_2', 'country_3']:
        country_path = os.path.join(TEST_DATA_DIR, country, 'images')
        if os.path.exists(country_path):
            image_files = glob.glob(os.path.join(country_path, '*.jpg'))
            for img_path in image_files:
                all_images.append({
                    'path': img_path,
                    'country': country,
                    'filename': os.path.basename(img_path)
                })
    
    return all_images

def process_image_and_save(image_info, model, output_dir):
    """
    Process a single image with Roboflow model and save results in YOLO format
    """
    image_path = image_info['path']
    filename = image_info['filename']
    
    try:
        # Run inference
        results = model.infer(image_path)
        
        # Convert to YOLO format
        yolo_string = convert_results_to_yolo_format(results)
        
        # Prepare output txt file path (same name as image but with .txt extension)
        txt_filename = os.path.splitext(filename)[0] + '.txt'
        output_txt_path = os.path.join(output_dir, txt_filename)
        
        # Save to file
        with open(output_txt_path, 'w') as f:
            f.write(yolo_string)
        
        # Count detections
        num_detections = len(results[0].predictions) if results and len(results) > 0 else 0
        
        return {
            'image_path': image_path,
            'output_path': output_txt_path,
            'num_detections': num_detections,
            'success': True
        }
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return {
            'image_path': image_path,
            'error': str(e),
            'success': False
        }

# Get all test images
test_images = get_all_test_images()
print(f"Total test images found: {len(test_images)}")

# Show distribution by country
for country in ['country_1', 'country_2', 'country_3']:
    country_count = len([img for img in test_images if img['country'] == country])
    print(f"{country}: {country_count} images")

Output directory: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result
Total test images found: 2961
country_1: 1024 images
country_2: 918 images
country_3: 1019 images


In [23]:
# Install tqdm if not available
try:
    from tqdm import tqdm
    print("tqdm is already available")
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm import tqdm
    print("tqdm installed successfully")

tqdm is already available


In [24]:
# Test processing on first 5 images
print("Testing batch processing on first 5 images...")

test_batch = test_images[:5]
test_results = []

for i, image_info in enumerate(test_batch):
    print(f"Processing test image {i+1}/5: {image_info['filename']}")
    result = process_image_and_save(image_info, model, OUTPUT_DIR)
    test_results.append(result)
    
    if result['success']:
        print(f"  ✓ Success: {result['num_detections']} detections saved to {os.path.basename(result['output_path'])}")
    else:
        print(f"  ✗ Failed: {result['error']}")

print(f"\nTest completed: {sum(1 for r in test_results if r['success'])}/5 images processed successfully")

Testing batch processing on first 5 images...
Processing test image 1/5: country1_008643.jpg
  ✓ Success: 1 detections saved to country1_008643.txt
Processing test image 2/5: country1_002089.jpg
  ✓ Success: 1 detections saved to country1_002089.txt
Processing test image 3/5: country1_007890.jpg
  ✓ Success: 1 detections saved to country1_007890.txt
Processing test image 4/5: country1_007682.jpg
  ✓ Success: 1 detections saved to country1_007682.txt
Processing test image 5/5: country1_008386.jpg
  ✓ Success: 3 detections saved to country1_008386.txt

Test completed: 5/5 images processed successfully


In [25]:
# Run batch processing on all test images
print("Starting batch inference on all test images...")
print(f"Processing {len(test_images)} images...")

inference_results = []
failed_images = []

# Process images with progress bar
for i, image_info in enumerate(tqdm(test_images, desc="Processing images")):
    result = process_image_and_save(image_info, model, OUTPUT_DIR)
    
    if result['success']:
        inference_results.append(result)
    else:
        failed_images.append(result)
    
    # Print progress every 100 images
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(test_images)} images...")

print(f"\nBatch inference completed!")
print(f"Successfully processed: {len(inference_results)} images")
print(f"Failed to process: {len(failed_images)} images")

# Print summary statistics
total_detections = sum(result['num_detections'] for result in inference_results)
print(f"Total detections found: {total_detections}")
print(f"Average detections per image: {total_detections / len(inference_results):.2f}")

# Show some examples of processed files
print("\nFirst 5 processed files:")
for i, result in enumerate(inference_results[:5]):
    print(f"{i+1}. {os.path.basename(result['output_path'])} - {result['num_detections']} detections")

Starting batch inference on all test images...
Processing 2961 images...


Processing images:   3%|▎         | 100/2961 [00:54<30:13,  1.58it/s]

Processed 100/2961 images...


Processing images:   5%|▍         | 137/2961 [01:16<26:09,  1.80it/s]


KeyboardInterrupt: 

In [26]:
# Check current progress and verify output format
import os

# Check how many files were created
existing_files = glob.glob(os.path.join(OUTPUT_DIR, "*.txt"))
print(f"Number of txt files created so far: {len(existing_files)}")

# Display content of a few sample files to verify format
if existing_files:
    print("\nSample output files:")
    for i, file_path in enumerate(existing_files[:3]):
        filename = os.path.basename(file_path)
        print(f"\n{filename}:")
        with open(file_path, 'r') as f:
            content = f.read().strip()
            if content:
                print(content)
            else:
                print("(empty file - no detections)")

# Show the expected format
print("\nExpected format:")
print("<class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>")
print("Example: 1 0.681293 0.833369 0.620362 0.331212 0.883995")

Number of txt files created so far: 137

Sample output files:

country1_007565.txt:
1 0.457480 0.718520 0.816171 0.557310 0.885512
3 0.699893 0.705429 0.320203 0.546879 0.870557

country1_000174.txt:
0 0.416476 0.668108 0.145194 0.075366 0.646091
0 0.337838 0.755337 0.125706 0.065342 0.445005
0 0.661816 0.715498 0.330131 0.142162 0.425606
0 0.180769 0.693920 0.327082 0.118310 0.417967

country1_002470.txt:
3 0.652128 0.947302 0.053931 0.104635 0.781448

Expected format:
<class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>
Example: 1 0.681293 0.833369 0.620362 0.331212 0.883995


In [ ]:
# Resumable batch processing - skip already processed images
def get_remaining_images(all_images, output_dir):
    """Get list of images that haven't been processed yet"""
    existing_txt_files = set()
    for txt_file in glob.glob(os.path.join(output_dir, "*.txt")):
        base_name = os.path.splitext(os.path.basename(txt_file))[0]
        existing_txt_files.add(base_name)
    
    remaining_images = []
    for img_info in all_images:
        img_base_name = os.path.splitext(img_info['filename'])[0]
        if img_base_name not in existing_txt_files:
            remaining_images.append(img_info)
    
    return remaining_images

# Get remaining images to process
remaining_images = get_remaining_images(test_images, OUTPUT_DIR)
print(f"Already processed: {len(test_images) - len(remaining_images)} images")
print(f"Remaining to process: {len(remaining_images)} images")

if len(remaining_images) > 0:
    print("\nTo continue processing, run the batch processing loop with remaining_images instead of test_images")
    print("You can also process in smaller batches to avoid interruption")
else:
    print("All images have been processed!")